# Figure 4 Absolute-Profile Sensitivity Analysis

This notebook asks whether feature neighborhoods are cleaner when Graphical Lasso profiles are represented by signed partial correlations or absolute partial correlations.

Signed profiles preserve whether conditional associations are positive or negative. Absolute profiles focus on dependency-neighborhood strength regardless of sign, which may be more relevant when comparing whether synthetic data preserves real structural neighborhoods.

## Setup

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from IPython.display import display

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]

pkg_root = repo_root / "data_synthesis"
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

import src.figure4_neighborhood as figure4_neighborhood
figure4_neighborhood = importlib.reload(figure4_neighborhood)

import src.main_figures_revision as figrev
figrev = importlib.reload(figrev)

EXPORT_DIR = repo_root / "data_synthesis" / "notebooks" / "revision_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {figrev.RUN_MODE} | CVAE_EPOCHS={figrev.CVAE_EPOCHS}")

In [ ]:
datasets = figrev.require_datasets()
figure4_real_data, figure4_synthetic_data, figure4_feature_names = figrev._get_figure4_precision_inputs()

pd.DataFrame({
    "dataset": list(figure4_real_data),
    "n_samples": [figure4_real_data[ds].shape[0] for ds in figure4_real_data],
    "n_features": [figure4_real_data[ds].shape[1] for ds in figure4_real_data],
    "alpha": [figrev.FIGURE4_ALPHAS[ds] for ds in figure4_real_data],
})

## Profile Representations

Both representations start from the real Graphical Lasso partial-correlation matrix.

- `signed`: cluster rows of the signed partial-correlation matrix.
- `absolute`: cluster rows after taking `abs(partial_corr)`, so sign is ignored and dependency-neighborhood strength dominates.

In [ ]:
def real_partial_corr(dataset_name):
    X = np.asarray(figure4_real_data[dataset_name], dtype=float)
    theta = figure4_neighborhood.fit_glasso_precision(X, figrev.FIGURE4_ALPHAS[dataset_name])
    return figure4_neighborhood.precision_to_partial_corr(theta)


def profile_matrix(partial, representation="signed"):
    profiles = np.asarray(partial, dtype=float).copy()
    np.fill_diagonal(profiles, 0.0)
    if representation == "absolute":
        profiles = np.abs(profiles)
    elif representation != "signed":
        raise ValueError("representation must be 'signed' or 'absolute'")
    return StandardScaler().fit_transform(profiles)


def heuristic_k(n_features, max_clusters=7):
    k = int(min(max_clusters, max(3, round(np.sqrt(n_features) * 1.35))))
    return min(k, max(2, n_features // 2))


def cluster_labels(profiles, metric="cosine", linkage_method="average", max_clusters=7):
    k = heuristic_k(profiles.shape[0], max_clusters=max_clusters)
    distances = pdist(profiles, metric=metric)
    linkage = hierarchy.linkage(distances, method=linkage_method)
    return hierarchy.fcluster(linkage, t=k, criterion="maxclust")


def cluster_summary_for(dataset_name, representation="signed", metric="cosine", linkage_method="average"):
    partial = real_partial_corr(dataset_name)
    profiles = profile_matrix(partial, representation=representation)
    labels = cluster_labels(profiles, metric=metric, linkage_method=linkage_method)
    counts = pd.Series(labels).value_counts().sort_values(ascending=False).to_list()
    sil = silhouette_score(profiles, labels, metric=metric) if 1 < len(set(labels)) < profiles.shape[0] else np.nan
    return {
        "dataset": dataset_name,
        "representation": representation,
        "metric": metric,
        "linkage": linkage_method,
        "requested_k": heuristic_k(profiles.shape[0]),
        "actual_k": len(set(labels)),
        "largest_cluster": counts[0],
        "cluster_sizes": ",".join(map(str, counts)),
        "silhouette": sil,
    }

## Signed vs Absolute, Same Distance

This is the core comparison: same average linkage and same distance metric, but signed vs absolute Graphical Lasso profiles.

In [ ]:
REPRESENTATION_METRIC = "cosine"  # Try: "cosine", "euclidean", "correlation"
REPRESENTATION_LINKAGE = "average"

signed_absolute_summary = pd.DataFrame([
    cluster_summary_for(dataset_name, representation=representation, metric=REPRESENTATION_METRIC, linkage_method=REPRESENTATION_LINKAGE)
    for dataset_name in figrev.DATASET_ORDER
    for representation in ["signed", "absolute"]
])

display(signed_absolute_summary)

## Full Representation x Distance Sensitivity Table

This table tries signed and absolute profile representations against Euclidean, cosine, and correlation distances. Use silhouette, cluster sizes, and biological interpretability together; do not optimize silhouette alone.

In [ ]:
rows = []
label_cache = {}
for dataset_name in figrev.DATASET_ORDER:
    partial = real_partial_corr(dataset_name)
    for representation in ["signed", "absolute"]:
        profiles = profile_matrix(partial, representation=representation)
        for metric in ["euclidean", "cosine", "correlation"]:
            labels = cluster_labels(profiles, metric=metric, linkage_method="average")
            counts = pd.Series(labels).value_counts().sort_values(ascending=False).to_list()
            sil = silhouette_score(profiles, labels, metric=metric) if 1 < len(set(labels)) < profiles.shape[0] else np.nan
            key = (dataset_name, representation, metric)
            label_cache[key] = labels
            rows.append({
                "dataset": dataset_name,
                "representation": representation,
                "metric": metric,
                "linkage": "average",
                "requested_k": heuristic_k(profiles.shape[0]),
                "actual_k": len(set(labels)),
                "largest_cluster": counts[0],
                "cluster_sizes": ",".join(map(str, counts)),
                "silhouette": sil,
            })

representation_distance_sensitivity = pd.DataFrame(rows)
for dataset_name in figrev.DATASET_ORDER:
    baseline = label_cache[(dataset_name, "signed", "cosine")]
    mask = representation_distance_sensitivity["dataset"] == dataset_name
    representation_distance_sensitivity.loc[mask, "ari_to_signed_cosine"] = representation_distance_sensitivity.loc[mask].apply(
        lambda row: adjusted_rand_score(baseline, label_cache[(row["dataset"], row["representation"], row["metric"])]),
        axis=1,
    )

out_path = EXPORT_DIR / "figure4_signed_absolute_profile_sensitivity.csv"
representation_distance_sensitivity.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

display(representation_distance_sensitivity.sort_values(["dataset", "silhouette"], ascending=[True, False]))

## Visualize One Dataset: Signed vs Absolute Profiles

This uses the same t-SNE layout approach for both representations, with point color showing the cluster labels from the selected representation. If absolute profiles help, you should see less fragmentation or more interpretable neighborhoods.

In [ ]:
VIS_DATASET = "HIV"  # Options: "HIV", "Breast Cancer", "Diabetes"
VIS_METRIC = "cosine"
VIS_LINKAGE = "average"
VIS_PERPLEXITY = 30
LABEL_TOP_N = 10

import matplotlib.pyplot as plt

partial = real_partial_corr(VIS_DATASET)
plot_rows = []
summary_rows = []
for representation in ["signed", "absolute"]:
    profiles = profile_matrix(partial, representation=representation)
    labels = cluster_labels(profiles, metric=VIS_METRIC, linkage_method=VIS_LINKAGE)
    kwargs = dict(
        n_components=2,
        perplexity=min(VIS_PERPLEXITY, max(1, profiles.shape[0] - 1)),
        init="pca",
        learning_rate="auto",
        random_state=figrev.SEED,
        metric="euclidean",
    )
    try:
        coords = TSNE(max_iter=1500, **kwargs).fit_transform(profiles)
    except TypeError:
        coords = TSNE(n_iter=1500, **kwargs).fit_transform(profiles)

    degrees = (np.abs(partial) > 1e-7).sum(axis=1)
    strengths = np.abs(partial).sum(axis=1)
    for feature_index, (x, y) in enumerate(coords):
        plot_rows.append({
            "representation": representation,
            "feature_index": feature_index,
            "feature": figure4_feature_names[VIS_DATASET][feature_index],
            "tsne_1": x,
            "tsne_2": y,
            "cluster_id": int(labels[feature_index]),
            "profile_cluster": f"cluster {int(labels[feature_index])}",
            "real_degree": float(degrees[feature_index]),
            "real_strength": float(strengths[feature_index]),
        })

    for cluster_id in sorted(set(labels)):
        idx = np.where(labels == cluster_id)[0]
        prominent_idx = idx[np.argsort(-strengths[idx])[:3]]
        summary_rows.append({
            "representation": representation,
            "cluster_id": int(cluster_id),
            "n_features": int(len(idx)),
            "prominent_features": ", ".join(figure4_feature_names[VIS_DATASET][i] for i in prominent_idx),
            "mean_degree": float(np.mean(degrees[idx])),
            "mean_strength": float(np.mean(strengths[idx])),
        })

visual_df = pd.DataFrame(plot_rows)
visual_summary = pd.DataFrame(summary_rows)

cluster_ids = sorted(visual_df["cluster_id"].unique())
colors = dict(zip(cluster_ids, plt.get_cmap("tab10")(np.linspace(0, 1, max(len(cluster_ids), 1)))))
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.8), constrained_layout=False)

for ax, representation in zip(axes, ["signed", "absolute"]):
    sub = visual_df[visual_df["representation"] == representation]
    for cluster_id, group in sub.groupby("cluster_id", sort=True):
        ax.scatter(
            group["tsne_1"],
            group["tsne_2"],
            s=35 + 6 * np.sqrt(group["real_strength"] + 1),
            color=colors[cluster_id],
            edgecolor="#222222",
            linewidth=0.45,
            alpha=0.88,
            label=f"cluster {cluster_id}",
        )
    label_rows = sub.sort_values("real_strength", ascending=False).head(LABEL_TOP_N)
    for _, row in label_rows.iterrows():
        ax.text(
            row["tsne_1"],
            row["tsne_2"],
            figure4_neighborhood._short_label(row["feature"], 14),
            fontsize=7,
            ha="center",
            va="center",
        )
    ax.set_title(f"{representation.title()} profiles", fontsize=12.5, weight="semibold")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.9)
        spine.set_edgecolor("#333333")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.01, 0.5), fontsize=8, title="Profile cluster")
fig.suptitle(
    f"{VIS_DATASET}: signed vs absolute Graphical Lasso profiles ({VIS_LINKAGE}+{VIS_METRIC}; perplexity={VIS_PERPLEXITY})",
    fontsize=13.5,
    weight="semibold",
    y=0.98,
)
fig.subplots_adjust(left=0.04, right=0.86, top=0.88, bottom=0.06, wspace=0.12)
plt.show()

display(visual_summary.sort_values(["representation", "cluster_id"]))


## Is This Graph Community Detection?

No. This notebook still performs profile clustering: each feature is represented by a row/fingerprint of the partial-correlation matrix.

Graph community detection would instead treat features as nodes and partial correlations as weighted edges, then find densely connected modules in that graph. That asks a different question: `same network module` rather than `similar connection fingerprint`.